# Sim 1: Forward kinematics, with the simulator as referee

**Roles today.** The **Driver** says the next step out loud. The **Keyboard**
types only what someone else has said. The **Checker** owns the board.

**The rule.** Every cell that moves the robot is preceded by a 🔮 PREDICT. Before
anyone runs it, the Checker writes the group's prediction on the board. After it
runs, the group says what matched, what did not, and why, before moving on.
Run every cell, in order, with Shift+Enter.

**The clock.** Part 0 by 0:08, Part A by 0:20, B1 by 0:38, B2 by 0:56, B3 by 1:06,
then Part C together. If you are behind, skip to Part C when the instructor calls it; the
submission finishes what you started.

**The pieces.** Three things work together today:

- **CoppeliaSim** runs the physics: the UR5's motors, its links, gravity. It must
  be open before you run the first cell. Your code talks to it over the network.
- **`screws`** is the course's robotics library: the functions of *Modern
  Robotics* in Python, under the names the notes use (`screws.fk_space` is the
  space form of the product of exponentials, and so on). We always write
  `screws.` in front of its functions so you can see where they come from.
- **`me465`** is this lab folder's own small helper module. It knows where the
  course scene is and how to open and close it: three functions, explained in
  the first cell.


In [ ]:
# the next line makes figures interactive: drag to rotate, instead of static pictures
%matplotlib widget
import numpy as np      # arrays and linear algebra; np.array, np.cross, np.radians, ...
import screws           # the course's robotics library (Modern Robotics in Python)
from me465 import open_lab, settle, close   # the three course helpers:
#   open_lab()          connects to the running CoppeliaSim, loads the course scene
#                       (a UR5 bolted to a 0.4 m pedestal) and starts the simulation.
#                       It returns two objects, `scene` and `arm`, used everywhere below.
#   settle(scene, arm)  steps the simulation forward until the joints stop moving
#                       (the simulated motors take time to get where they were sent),
#                       and returns the joint angles they stopped at.
#   close(scene)        stops the simulation and disconnects, at the end of the day.

np.set_printoptions(precision=4, suppress=True)   # print 4 decimals, no scientific notation

scene, arm = open_lab()
#   scene: the simulation. scene.step() advances it one time step (0.05 s).
#   arm:   the UR5. arm.command_positions(theta) sends its six joints to the angles in
#          theta (radians); arm.theta() reads the angles they are at; arm.tip_frame()
#          reads the 4x4 configuration of the tool frame; arm.joint_frames() reads the
#          six joint frames. All frames come back expressed in the simulator's world frame.
print(arm)


## Part 0: Move the arm (warm-up)

Before any theory, drive the robot. The cell below sends six joint angles to
the simulated UR5 and waits for it to get there. Run it, look at CoppeliaSim,
then **change the angles and run it again**, several times.

🔮 **PREDICT.** Before each run, say which joint you changed and what you
expect to move: the whole arm, the forearm, just the wrist? Positive angles
follow the right-hand rule about each joint's axis; find out which way that
is for each joint.


In [ ]:
theta_command = np.radians([0, 0, 0, 0, 0, 0])   # six joint angles, in degrees; edit me and re-run
arm.command_positions(theta_command)             # send them to the simulated motors
measured = settle(scene, arm)                    # step the simulation until the joints stop moving
print("joints stopped at (deg):", np.round(np.degrees(measured), 1))
print("tool frame, in the simulator's world frame:\n", arm.tip_frame())


**Sliders.** The same thing without retyping: one slider per joint. Drag a
slider and release it; the arm follows and the tool frame is printed. Two
minutes with these and you will know the UR5's joints by name. (Pass
`continuous=True` for updates while you drag, at the cost of speed.)


In [ ]:
from me465 import joint_sliders

sliders = joint_sliders(scene, arm)


## Part A: The book's model of the UR5 (every group in step)

The notes give MR's model of the UR5 (MR Example 4.5): its home configuration
$M$ and its space-frame screw axes $\mathcal{S}_1, \dots, \mathcal{S}_6$, from
the rounded dimensions in this figure.

![The UR5 at its zero position, as a skeleton. Blue pins are joint axes, with the positive direction of each screw axis marked.](ur5-skeleton.png)

Each screw axis is a direction and a point. For a revolute joint,
$\mathcal{S}_i = (\hat\omega_i, v_i)$ with $\hat\omega_i$ the unit vector along
the joint axis and $v_i = -\hat\omega_i \times q_i$ for any point $q_i$ on the
axis (notes, Definition 3.12). The library builds one for you:
`screws.revolute_axis(q, s_hat)` returns $(\hat s, -\hat s \times q)$. Its
argument is called `s_hat` because the textbook writes $\hat s$ for the
direction of a general screw; for a revolute joint that direction *is*
$\hat\omega$, so the two are the same thing here.

The model is typed in below. **Check one row** before running: pick a joint,
read its $\hat\omega_i$ and $q_i$ off the figure, and confirm the row.


In [ ]:
# the six dimensions of the figure, in metres
W1, W2, L1, L2, H1, H2 = 0.109, 0.082, 0.425, 0.392, 0.089, 0.095

# M: the tool frame {b} at the zero position, expressed in {s}. Its columns are the
# axes of {b} (x_b = -x_s, y_b = z_s, z_b = y_s) and its last column is the origin.
M = np.array([[-1, 0, 0, L1 + L2],
              [ 0, 0, 1, W1 + W2],
              [ 0, 1, 0, H1 - H2],
              [ 0, 0, 0, 1]])

# S: the six screw axes, one per column (that is how the book and the library store
# them, as a 6x6 matrix). Each is built from a point q on the axis and the axis
# direction; np.array([...]).T turns the six rows into six columns.
S = np.array([screws.revolute_axis(q=[0, 0, 0],             s_hat=[0, 0, 1]),   # joint 1: vertical, through the base
              screws.revolute_axis(q=[0, 0, H1],            s_hat=[0, 1, 0]),   # joint 2: along y, at height H1
              screws.revolute_axis(q=[L1, 0, H1],           s_hat=[0, 1, 0]),   # joint 3: same, L1 further along x
              screws.revolute_axis(q=[L1 + L2, 0, H1],      s_hat=[0, 1, 0]),   # joint 4: same, L2 further
              screws.revolute_axis(q=[L1 + L2, W1, 0],      s_hat=[0, 0, -1]),  # joint 5: points down, offset W1 in y
              screws.revolute_axis(q=[L1 + L2, 0, H1 - H2], s_hat=[0, 1, 0])]).T  # joint 6: along y, H2 lower

print("M =\n", M)
print("S (one screw axis per column; rows are omega_x, omega_y, omega_z, v_x, v_y, v_z) =\n", S)


### Two frames, and the pedestal

The simulator reports everything in its **world frame** $\{w\}$: the frame
CoppeliaSim itself uses, with its origin on the floor under the pedestal and
$z$ pointing up. The book's **space frame** $\{s\}$ sits at the base of the
robot, on top of the pedestal, with its axes parallel to $\{w\}$. The two
frames differ by a pure lift of $0.4$ m:

$$T_{ws} = \mathrm{Trans}(0, 0, 0.4), \qquad T_{sw} = T_{ws}^{-1},$$

where $T_{ws}$ is the configuration of $\{s\}$ in $\{w\}$. A tool
configuration the simulator reports, $T_{w,\text{tool}}$, becomes the one the
book would write by $T_{s,\text{tool}} = T_{sw}\,T_{w,\text{tool}}$ (subscript
cancellation, notes 3.4). Every simulator reading below goes through $T_{sw}$
before it is compared with the model.

### Two zero positions

Here is the catch of the day. The figure above shows the book's zero position:
all six $\theta_i = 0$ with the arm laid out along $\hat x_s$. The simulated
UR5 also has a zero position, all six of *its* joint angles zero, and **it is
not the same pose**: CoppeliaSim's model was built with its own idea of where
zero is. Both are legitimate. The product of exponentials only needs $M$ and
the screw axes to be measured in one agreed home pose; the book chose one, the
simulator's builders chose another.

🔮 **PREDICT.** Look at the CoppeliaSim window: the arm is sitting at its zero
position right now. Sketch it on the board, and mark where you think $\{s\}$
and the tool frame are. Then say where the tool frame's origin is in $\{s\}$,
roughly, in metres (the pedestal top is $z = 0$; the arm's links are about
$0.43$ and $0.39$ m long). Is that anywhere near the book's $M$? The cell
commands $\theta = 0$, which changes nothing, and reads the answer.


In [ ]:
# T_ws: the configuration of {s} in {w}, a pure translation of 0.4 m up.
# rp_to_transform(R, p) builds the 4x4 matrix from a rotation R and a position p.
T_ws = screws.rp_to_transform(np.eye(3), [0, 0, 0.4])
T_sw = screws.transform_inv(T_ws)      # the inverse: takes world coordinates to {s}

arm.command_positions(np.zeros(6))     # send all six joints to 0 rad
settle(scene, arm)                     # let the simulated motors get there

T_w_tool = arm.tip_frame()             # where the simulator says the tool is, in {w}
T_s_tool = T_sw @ T_w_tool             # the same, in {s}: now comparable with M
print("simulator, tool at theta = 0, in {s}:\n", T_s_tool)
print("the book's M:\n", M)


📺 **Screens up** when the instructor calls it. Everyone's robot should look
the same. Does it look like your sketch? Does `T_s_tool` look like `M`? Say in
one sentence how the simulator's zero position differs from the book's home
position.


## Part B: Build

### B1. Read the screw axes off the simulated robot

The product of exponentials works from *any* home configuration, as long as
$M$ and the screw axes are all measured in that same configuration. So take
the scene's own zero pose as the home, and read $M$ and every
$\mathcal{S}_i$ off the simulator.

In CoppeliaSim every joint carries a frame, and **a joint turns about its
frame's $z$-axis**. `arm.joint_frames()` returns the six joint frames, one
4x4 matrix each, in $\{w\}$; call the one for joint $i$ $T_{wi}$. Once it is
expressed in $\{s\}$ as $T_{si} = (R_{si}, p_{si})$:

- $\hat\omega_i$ is the third column of $R_{si}$ (the frame's $z$-axis);
- $q_i$, any point on the axis, can be the frame's origin $p_{si}$;
- $v_i = -\hat\omega_i \times q_i$.

The loop below has one job left for you: build the 6-vector for each joint.

🔮 **PREDICT.** Before you run it: which columns of `S_scene` will match the
book's `S`, and which will not? Why?


In [ ]:
arm.command_positions(np.zeros(6))     # the scene's zero pose is the home we measure in
settle(scene, arm)

S_cols = []                            # will hold six 6-vectors, one per joint
frames_home = []                       # and the six joint frames in {s}, for the picture below
for T_wi in arm.joint_frames():        # joint i's frame, in {w}, one joint at a time
    T_si = T_sw @ T_wi                 # the same frame in {s}
    frames_home.append(T_si)
    R_si, p_si = screws.transform_to_rp(T_si)   # its rotation matrix and its origin
    # TODO: omega is the third column of R_si, R_si[:, 2]; q is p_si; v = -omega x q.
    #       np.cross(a, b) is the cross product and np.concatenate([a, b]) joins two
    #       3-vectors into a 6-vector. Append that 6-vector to S_cols.
    pass

if len(S_cols) != 6:
    raise RuntimeError("S_cols is empty: fill in the TODO in the loop above, one 6-vector per joint")
S_scene = np.array(S_cols).T           # six columns, like S
M_scene = T_sw @ arm.tip_frame()       # the tool frame at the scene's zero pose, in {s}

print("S_scene (the scene's screw axes at its zero pose) =\n", S_scene)
print("M_scene =\n", M_scene)
print("column-by-column, does omega match the book's S?  ",
      [bool(np.allclose(S_scene[:3, i], S[:3, i], atol=1e-3)) for i in range(6)])


📈 **Picture it (optional, if you are ahead).** `screws.viz` draws the arm at
its zero pose with the axes you just read off the simulator, each in its own
colour. Drag the figure to rotate it, and use the toolbar on its left to zoom or save it. Does each $\mathcal{S}_i$ pass
through its joint, and is $\hat\omega_i$ the joint's $z$-axis? Add
`frame="body"` to see the same robot from $\{b\}$, and `joints=[2],
show=("skeleton", "frames", "axes", "omega", "construction")` to see how
$v_3 = -\hat\omega_3 \times q_3$ is built.


In [ ]:
from screws import viz

# the scene's model as a Robot: its M, its screw axes, and where its joints are at home
robot_scene = screws.Robot.from_screw_axes(M_scene, list(S_scene.T), joint_frames_home=frames_home,
                                           name="UR5 (scene)")
viz.draw_robot(robot_scene);


### B2. Your configuration card

Your group's card gives six joint angles in degrees. For the card:

1. 🔮 **PREDICT, roughly.** Sketch the arm on the board. Where is the tool?
2. 🔮 **PREDICT, exactly.** The cell computes $T_{s,\text{tool}}$ with
   `screws.fk_space(M_scene, S_scene, theta)`, the space form of the product of
   exponentials (MR's `FKinSpace`) using the scene's own model. Before running,
   say which entries of the result you can predict from the sketch.
3. **Run.** The cell then commands the angles, lets the arm settle, and reads
   the ground truth from the simulator.
4. **Explain the gap.** `pose_error` reports how far apart two configurations
   are: the distance between their origins in millimetres, and the angle of
   the rotation that takes one orientation to the other in degrees. Where does
   what is left come from?


In [ ]:
card = [0, 0, 0, 0, 0, 0]              # TODO: your card, six angles in degrees
theta = np.radians(card)               # the library works in radians

# the prediction: forward kinematics with the scene's own model
T_pred = screws.fk_space(M_scene, S_scene, theta)

# the ground truth: command the angles, wait, read the tool frame in {s}
arm.command_positions(theta)
measured = settle(scene, arm)          # the angles the joints actually stopped at
T_true = T_sw @ arm.tip_frame()


def pose_error(T_a, T_b):
    """How far apart two configurations are: (position error in mm, rotation error in degrees)."""
    dp = np.linalg.norm(T_a[:3, 3] - T_b[:3, 3]) * 1000          # distance between the origins
    R = T_a[:3, :3].T @ T_b[:3, :3]                               # the rotation from one to the other
    dth = np.degrees(np.arccos(np.clip((np.trace(R) - 1) / 2, -1, 1)))   # its angle
    return dp, dth


print("predicted:\n", T_pred)
print("simulator:\n", T_true)
print("error: %.2f mm, %.3f deg" % pose_error(T_pred, T_true))
print("measured angles minus commanded (deg):", np.degrees(measured - theta))


### B3. The book's model, driven through the offset

In the book's zero position (the figure in Part A, MR Figure 4.6), the upper
arm and forearm lie along $\hat x_s$ at height $H_1$, so the tool ends up
almost level with the base, $0.8$ m out. In the scene's zero position the
same two links stand straight up. The six angles that turn the simulated arm
from its zero position into the book's are a fact about the scene, given here:

$$\theta_\text{offset} = (-90^\circ,\ 90^\circ,\ 0,\ 90^\circ,\ 0,\ 90^\circ).$$

So a joint vector for the book's model and one for the scene are related by
$\theta_\text{scene} = \theta_\text{book} + \theta_\text{offset}$, and the
book's model at $\theta_\text{book}$ should land where the simulator puts the
robot at $\theta_\text{scene}$.

🔮 **PREDICT.** First for the home pose itself: the cell commands the offset
alone. How close to $M$ will the tool be, in millimetres and degrees? Then for
your card: how close will the book's model come to the simulator?


In [ ]:
offset = np.radians([-90, 90, 0, 90, 0, 90])

# the home pose: the offset alone should put the tool at M
arm.command_positions(offset)
settle(scene, arm)
T_home = T_sw @ arm.tip_frame()
print("simulator at the offset, in {s}:\n", T_home)
print("the book's M:\n", M)
print("error: %.2f mm, %.3f deg" % pose_error(T_home, M))

# your card, through the book's model: theta_book = theta_scene - offset
theta_book = theta - offset
T_book = screws.fk_space(M, S, theta_book)
arm.command_positions(theta)
settle(scene, arm)
print("card, book's model vs simulator: %.2f mm, %.3f deg" % pose_error(T_book, T_sw @ arm.tip_frame()))


When the orientation matches, look at what is left in the position: a few
millimetres, always in the same direction. Is that a mistake, or are two
models of the same robot really different? Say on the board which dimension
of the figure you suspect, and how you would find out. The submission asks
you to.


## Part C: Converge

Wait for the instructor. Every group commands the same angles at the same
moment, and we compare the robots side by side.


In [ ]:
converge = np.radians([0, 0, 0, 0, 0, 0])   # the instructor gives these in class

arm.command_positions(converge)
settle(scene, arm)
print("tool, in {s}:\n", T_sw @ arm.tip_frame())


When you are done for the day:

In [ ]:
close(scene)